In [3]:
print("ok")

ok


In [4]:
%pwd

'c:\\Users\\sharm\\Desktop\\DSProjects\\genai-agentiai-nlp\\FinChatProject\\research'

In [5]:
import os
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\sharm\\Desktop\\DSProjects\\genai-agentiai-nlp\\FinChatProject'

In [11]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

#### Extract text from PDF files

In [17]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [18]:
extract_data = load_pdf_files("data")

In [10]:
extract_data

[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data\\merged_AI_tech_reports.pdf', 'total_pages': 1459, 'page': 0, 'page_label': '1'}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n___________________________________________\nFORM 10-K\n___________________________________________\n(Mark One)\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended December 31, 2024OR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from              to             .\nCommission file number: 001-37580\n___________________________________________\nAlphabet Inc.\n(Exact name of registrant as specified in its charter)\n___________________________________________\nDelaware 61-1767919\n(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)\n1600 Amphith

In [19]:
len(extract_data)

1459

In [ ]:
# It will only return source and the page content of the source
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only "source" in metadata and the orginal page_content
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [21]:
minimal_docs = filter_to_minimal_docs(extract_data)

In [22]:
minimal_docs

[Document(metadata={'source': 'data\\merged_AI_tech_reports.pdf'}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n___________________________________________\nFORM 10-K\n___________________________________________\n(Mark One)\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended December 31, 2024OR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from              to             .\nCommission file number: 001-37580\n___________________________________________\nAlphabet Inc.\n(Exact name of registrant as specified in its charter)\n___________________________________________\nDelaware 61-1767919\n(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)\n1600 Amphitheatre ParkwayMountain View, CA 94043\n(Address of principal executive offices, including zip code)\n(650) 253-00

#### Chunking

In [23]:
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
        length_function = len
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [24]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 13186


In [22]:
texts_chunk

[Document(metadata={'source': 'data\\merged_AI_tech_reports.pdf'}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n___________________________________________\nFORM 10-K\n___________________________________________\n(Mark One)\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended December 31, 2024OR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from              to             .\nCommission file number: 001-37580'),
 Document(metadata={'source': 'data\\merged_AI_tech_reports.pdf'}, page_content="___________________________________________\nAlphabet Inc.\n(Exact name of registrant as specified in its charter)\n___________________________________________\nDelaware 61-1767919\n(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)\n1600 Amphitheatre ParkwayMountain View, C

Now this document has converted as chunks, and each chunk has 500 tokens

#### Embeddings

In [12]:
# Hugging face embedding model
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings

embedding = download_embeddings()

C:\Users\sharm\AppData\Local\Temp\ipykernel_34504\114716253.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [13]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [14]:
# Vector representation of this sentence

vector = embedding.embed_query("Hello world")
vector

[-0.03447727486491203,
 0.03102312609553337,
 0.006734980270266533,
 0.026108933612704277,
 -0.03936205804347992,
 -0.16030246019363403,
 0.06692394614219666,
 -0.006441438104957342,
 -0.047450482845306396,
 0.014758863486349583,
 0.07087534666061401,
 0.05552757531404495,
 0.019193356856703758,
 -0.02625126577913761,
 -0.01010954286903143,
 -0.026940442621707916,
 0.022307462990283966,
 -0.02222665585577488,
 -0.14969263970851898,
 -0.017493024468421936,
 0.007676282897591591,
 0.054352231323719025,
 0.0032544038258492947,
 0.03172588348388672,
 -0.08462139964103699,
 -0.029405992478132248,
 0.051595550030469894,
 0.048124078661203384,
 -0.003314835485070944,
 -0.05827915295958519,
 0.04196925833821297,
 0.022210702300071716,
 0.1281888633966446,
 -0.022338951006531715,
 -0.011656239628791809,
 0.06292837113142014,
 -0.03287634998559952,
 -0.09122604131698608,
 -0.03117534890770912,
 0.052699536085128784,
 0.04703483358025551,
 -0.08420310169458389,
 -0.030056182295084,
 -0.0207448396

In [15]:
# dimensions of the vector
print("Vector Length:", len(vector))

Vector Length: 384


#### Store this vector in pinecone database

In [5]:
# loading the env file
from dotenv import load_dotenv
import os
load_dotenv()

True

In [6]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Once got API_KEY, need to send as an environment
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [7]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [8]:
pc

In [34]:
# Create an index - means creating a database

from pinecone import ServerlessSpec

index_name = "finance-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,    # Dimension of the embeddings
        metric="cosine",   # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
# Storing the vector
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

NameError: name 'texts_chunk' is not defined

Successfully stored all of the chunks and it has also converted to the vector representation.

In [ ]:
## First I have created the object. Now, what if I want to Load the existing index in the pinecone and continue my work.
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name="finance-chatbot",
    embedding=embedding
)


In [40]:
# Now, if I want to add more data to the existing pinecone index. Suppose I got another PDF file and I want to store in the same index
# let's create a dummy document

dswith = Document(
    page_content="LSBCAGLOBE is a data driven research lab that provides research on AI.",
    metadata={"source":"lsbcaglobe.com"}
)


In [41]:
# This is the code
docsearch.add_documents(documents=[dswith])

['119224e7-ccb3-4c15-9c69-cb0f7352ed7f']

copy and search the id

#### Now, creating a retriever, means connecting with the LLM and user will ask the question and get the response.

In [17]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

k = 3 means it will return three relevant response. Three most similarity response from the knowledge base.

In [18]:
retrieved_docs = retriever.invoke("Microsoft revenue")
retrieved_docs

[Document(id='81579b46-8b02-4a34-97e3-b0d486428b32', metadata={'source': 'data\\merged_AI_tech_reports.pdf'}, page_content='Our Microsoft Cloud revenue, which includes Azure and other cloud services, Office 365 Commercial, the commercial portion of LinkedIn, Dynamics 365, and other commercial cloud properties, was $137.4 billion, $111.6 billion, and $91.4 billion in fiscal years 2024, 2023, and 2022, respectively. These amounts are primarily included in Server products and cloud services, Office products and cloud services, LinkedIn, and Dynamics products and cloud services in the table above.'),
 Document(id='30fac4dd-cef0-4f1e-8b29-3ee2a077b954', metadata={'source': 'data\\merged_AI_tech_reports.pdf'}, page_content='Our Microsoft Cloud revenue, which includes Microsoft 365 Commercial cloud, Azure and other cloud services, the commercial portion of LinkedIn, and Dynamics 365, was $42.4 billion and $122.2 billion for the three and nine months ended March 31, 2025, respectively, and $35

So, based on the score, it is actually returning the three relevant response from this particular question. Let's refine this repsonse with LLM. Now, lets connect the LLM.

In [20]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [21]:
# Now, let's create the langchain

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [22]:
system_prompt = (
    "You are an experienced financial assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "the question. If you don't know the answer, say that you"
    "don't know. Use three sentences maximum and keep the"
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [23]:
# Creating the chain
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [24]:
response = rag_chain.invoke({"input": "Compare the revenue of netflix and tesla"})
print(response["answer"])

I don't have access to Tesla's financial information, but for Netflix, the revenue for the most recent year ended December 31, 2024, was $39,000,966,000. You may need to look up Tesla's latest revenue figures to make the comparison.


In [25]:
response = rag_chain.invoke({"input": "Find me tesla revenue figures"})
print(response["answer"])

Tesla's total revenues for the years ended December 31 are as follows: $96,773 million in 2023, $81,462 million in 2022, and $53,823 million in 2021.


In [26]:
response = rag_chain.invoke({"input": "Find me netflix revenue for year 2023"})
print(response["answer"])

Netflix's revenue for the year 2023 was $33,723,297,000.


In [36]:
response = rag_chain.invoke({"input": "What are the key risks tied to ETFs and mutual fund flows?"})
print(response["answer"])

Key risks tied to ETFs include significant market volatility, deviations from expected trading prices, and complexities associated with certain asset classes like digital assets. For mutual fund flows, risks involve potential adverse effects on business, growth, and financial conditions stemming from unpredictable market events. Both investment types may be significantly impacted by the broader market environment and investor behavior.


This is the power of RAG

Now modular coding